In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import BayesianRidge
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']  
plt.rcParams['axes.unicode_minus'] = False  

In [ ]:

models = {
    '3.Ridge': Ridge(alpha=1.0, random_state=42),
    '4.KNN': KNeighborsRegressor(),
    '5.Gaussian Process': GaussianProcessRegressor(random_state=42),
    '6.DecisionTree': DecisionTreeRegressor(max_depth=5, random_state=42),
    '7.Neural Network': MLPRegressor(random_state=42, max_iter=1000),
    '8.RandomForest': RandomForestRegressor(n_estimators=100,random_state=42),
    '9.SVR': SVR(kernel='linear', C=1.0),
    '10.AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    '11.GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    '12.XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42),
    '13.CatBoost': CatBoostRegressor(n_estimators=100, random_state=42, 
                                  learning_rate=0.1,
                                  early_stopping_rounds=50,
                                  verbose=False),
    '14.LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1 )
}


In [ ]:
results = []
predictions_val = {}
predictions_test = {}

import os

target_models = ['4.KNN', '5.Gaussian Process', '7.Neural Network']
for name, model in models.items():
    if name not in target_models:
        file_path = os.path.join("data", f"{name}best.xlsx")
        data = pd.read_excel(file_path)
        X = data.iloc[:, :-1]  
        y = data.iloc[:, -1]   
        feature_names = list(X.columns)

        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
        y_val_pred = model.predict(X_val_scaled) 

    else:
        data = pd.read_excel('data/13.CatBoost best.xlsx') 
        X = data.iloc[:, :-1]  
        y = data.iloc[:, -1]   
        feature_names = list(X.columns)

        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
        y_val_pred = model.predict(X_val_scaled) 
        
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    results.append({
        'Model': name,
        'Train_RMSE': train_rmse,
        'val_RMSE' :val_rmse,
        'Test_RMSE': test_rmse,
        'Train_R2': train_r2,
        'val_R2' :val_r2,
        'Test_R2': test_r2
    })
print("ok")

In [ ]:
results_df = pd.DataFrame(results)

with pd.option_context('display.float_format', '{:.4f}'.format):
    print(results_df)

In [ ]:
display(results_df)